# **1. Data collection**

This notebook builds the raw layer of the project from **StatsBomb Open Data**, accessed through `statsbombpy`: matches and events of four major European leagues (La Liga, Premier League, Serie A, and Ligue 1) in the 2015/2016 season. The 1. Bundesliga is left out because StatsBomb Open Data only covers the 34 Bayer Leverkusen matches of that season.

| step | source | result |
|---|---|---|
| scope | `COMPETITIONS_SEASONS` | four competition–season pairs |
| matches | `sb.matches` | one file per competition and season in `data/raw/matches/` |
| events | `sb.events` | one file per match in `data/raw/events/` |
| consolidation | per-match files | `data/raw/combined/all_matches_combined.parquet` and `all_events_combined.parquet` |
| audit | files and combined tables | coverage and checks |

Files already on disk are reused, so the collection can be interrupted and resumed without repeating requests.

In [1]:
import os
import time
import warnings
from pathlib import Path
from typing import List, Tuple, Optional, Union

import numpy as np
import pandas as pd
from IPython.display import display
from statsbombpy import sb

warnings.filterwarnings('ignore')

### **1.1. Configuration**

The project root is detected automatically. `MATCHES_PATH` and `EVENTS_PATH` store one file per competition–season and per match; `COMBINED_DATA_PATH` receives the combined tables read by the analysis notebook. The identifiers in `COMPETITIONS_SEASONS` follow the StatsBomb Open Data catalogue (`sb.competitions()`).

In [2]:
# The project root is detected automatically, inside or outside `notebooks/`.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data"
RAW_DATA_PATH = DATA_PATH / "raw"
COMBINED_DATA_PATH = RAW_DATA_PATH / "combined"
VIZ_PATH = ROOT / "viz"
MATCHES_PATH = RAW_DATA_PATH / "matches"
EVENTS_PATH = RAW_DATA_PATH / "events"
for path in [MATCHES_PATH, EVENTS_PATH, COMBINED_DATA_PATH, VIZ_PATH]:
    path.mkdir(parents=True, exist_ok=True)

print("Data directories created:")
print(f"  Matches: {MATCHES_PATH}")
print(f"  Events: {EVENTS_PATH}")
print(f"  Combined: {COMBINED_DATA_PATH}")
print(f"  Viz: {VIZ_PATH}")


# StatsBomb Open Data (competition_id, season_id) pairs: 2015/2016 season.
COMPETITIONS_SEASONS = [
    (11, 27),  # La Liga
    (2, 27),   # Premier League
    (12, 27),  # Serie A
    (7, 27),   # Ligue 1
]

print(f"Competition–season pairs to collect: {len(COMPETITIONS_SEASONS)}")
print(f"Selected pairs: {COMPETITIONS_SEASONS}")

Data directories created:
  Matches: /Users/gabrielcarbinatto/Documents/GitHub/Synergy-VAEP/data/raw/matches
  Events: /Users/gabrielcarbinatto/Documents/GitHub/Synergy-VAEP/data/raw/events
  Combined: /Users/gabrielcarbinatto/Documents/GitHub/Synergy-VAEP/data/raw/combined
  Viz: /Users/gabrielcarbinatto/Documents/GitHub/Synergy-VAEP/viz
Competition–season pairs to collect: 4
Selected pairs: [(11, 27), (2, 27), (12, 27), (7, 27)]


## **2. Utility functions**

`generate_filename` sets the name of each file and `check_existing_files` finds what has already been downloaded. `fetch_with_retry` repeats failed calls with increasing waits. `normalize_match_id_columns` standardizes manager identifiers, which arrive with mixed types, and `ensure_coordinate_columns` splits the `x` and `y` coordinates of each location.

In [3]:
def generate_filename(competition_id: int, season_id: int, data_type: str = "matches") -> str:
    return f"{data_type}_competition_{competition_id}_season_{season_id}.parquet"


def check_existing_files(path: Path, prefix: str) -> List[Tuple[int, int]]:
    existing_pairs = []
    
    for filename in os.listdir(path):
        if filename.endswith('.parquet') and filename.startswith(prefix):
            parts = filename.replace('.parquet', '').split('_')
            if len(parts) >= 5:
                try:
                    competition_id = int(parts[2])
                    season_id = int(parts[4])
                    existing_pairs.append((competition_id, season_id))
                except (ValueError, IndexError):
                    print(f"Warning: invalid file name format: {filename}")
    
    return existing_pairs


def fetch_with_retry(fetch_function, *args, max_retries: int = 3, **kwargs) -> Optional[pd.DataFrame]:
    for attempt in range(max_retries):
        try:
            data = fetch_function(*args, **kwargs)
            return data
        except Exception as e:
            if attempt < max_retries - 1:
                wait_time = (attempt + 1) * 2
                print(f"  Attempt {attempt + 1} failed: {e}")
                print(f"  Retrying in {wait_time} seconds...")
                time.sleep(wait_time)
            else:
                print(f"  Failed after {max_retries} attempts: {e}")
                return None
    
    return None


MATCH_ID_COLUMNS = [
    'home_manager_id',
    'home_manager_country_id',
    'away_manager_id',
    'away_manager_country_id',
]


def normalize_match_id_columns(matches_df: pd.DataFrame) -> pd.DataFrame:
    normalized = matches_df.copy()

    for column in MATCH_ID_COLUMNS:
        if column not in normalized.columns:
            continue

        def normalize_id(value):
            if pd.isna(value):
                return pd.NA
            if isinstance(value, (float, np.floating)) and value.is_integer():
                return str(int(value))
            return str(value)

        normalized[column] = normalized[column].map(normalize_id).astype('string')

    return normalized


def ensure_coordinate_columns(events_df: pd.DataFrame) -> pd.DataFrame:
    coord_specs = [
        ('location', ('location_x', 'location_y')),
        ('pass_end_location', ('pass_end_location_x', 'pass_end_location_y')),
        ('carry_end_location', ('carry_end_location_x', 'carry_end_location_y')),
    ]
    for src_col, (x_col, y_col) in coord_specs:
        if src_col not in events_df.columns:
            continue
        needs_backfill = (
            x_col not in events_df.columns or
            y_col not in events_df.columns or
            events_df[[x_col, y_col]].isna().all().all()
        )
        if needs_backfill:
            events_df[[x_col, y_col]] = events_df[src_col].apply(
                lambda x: pd.Series(x[:2]) if isinstance(x, (tuple, list, np.ndarray)) else pd.Series([np.nan, np.nan])
            )
    return events_df

## **3. Matches**

The matches of each competition–season pair are downloaded once and saved to `data/raw/matches/`. The combined table is rewritten on every run.

In [4]:
def collect_match_data() -> pd.DataFrame:
    existing_matches = check_existing_files(MATCHES_PATH, "matches_competition")
    print(f"Found {len(existing_matches)} existing match files.")
    
    all_matches_dfs = []
    
    for competition_id, season_id in COMPETITIONS_SEASONS:
        if (competition_id, season_id) in existing_matches:
            print(f"\nLoading existing data: competition {competition_id}, season {season_id}")
            parquet_path = MATCHES_PATH / generate_filename(competition_id, season_id)
            matches_df = pd.read_parquet(parquet_path)
            matches_df = normalize_match_id_columns(matches_df)
            all_matches_dfs.append(matches_df)
        else:
            print(f"\nFetching new data: competition {competition_id}, season {season_id}")
            matches_df = fetch_with_retry(
                sb.matches,
                competition_id=competition_id,
                season_id=season_id
            )
            
            if matches_df is not None:
                matches_df = normalize_match_id_columns(matches_df)
                parquet_path = MATCHES_PATH / generate_filename(competition_id, season_id)
                matches_df.to_parquet(parquet_path, index=False)
                print(f"  Saved to: {parquet_path.name}")
                all_matches_dfs.append(matches_df)
            
            time.sleep(1) 
    
    if all_matches_dfs:
        all_matches = pd.concat(all_matches_dfs, ignore_index=True)
        all_matches = normalize_match_id_columns(all_matches)
        print(f"\nMatches collected: {len(all_matches)}")
        
        combined_path = COMBINED_DATA_PATH / "all_matches_combined.parquet"
        all_matches.to_parquet(combined_path, index=False)
        print(f"Combined file saved to: {combined_path}")
        
        return all_matches
    else:
        print("\nNo match data collected")
        return pd.DataFrame()


all_matches = collect_match_data()

Found 5 existing match files.

Loading existing data: competition 11, season 27

Loading existing data: competition 2, season 27

Loading existing data: competition 12, season 27

Loading existing data: competition 7, season 27

Matches collected: 1517
Combined file saved to: /Users/gabrielcarbinatto/Documents/GitHub/Synergy-VAEP/data/raw/combined/all_matches_combined.parquet


## **4. Events**

Events are downloaded match by match, with a one-second pause between requests, and saved to `data/raw/events/`. The combined table is sorted by match, period, minute, second, and event index.

In [5]:
def collect_event_data(match_ids: List[int]) -> pd.DataFrame:
    existing_events = []
    for filename in os.listdir(EVENTS_PATH):
        if filename.endswith('.parquet') and filename.startswith('events_match_'):
            try:
                match_id = int(filename.replace('events_match_', '').replace('.parquet', ''))
                existing_events.append(match_id)
            except ValueError:
                pass
    
    print(f"Found {len(existing_events)} existing event files.")
    
    all_events_dfs = []
    total_matches = len(match_ids)
    
    for idx, match_id in enumerate(match_ids):
        if match_id in existing_events:
            events_path = EVENTS_PATH / f"events_match_{match_id}.parquet"
            events_df = pd.read_parquet(events_path)
            all_events_dfs.append(events_df)
        else:
            print(f"\nFetching events for match {match_id}")
            events_df = fetch_with_retry(sb.events, match_id=match_id)
            
            if events_df is not None:
                events_path = EVENTS_PATH / f"events_match_{match_id}.parquet"
                events_df.to_parquet(events_path, index=False)
                all_events_dfs.append(events_df)
            
            time.sleep(1) 
        
        if (idx + 1) % 50 == 0:
            print(f"\nProgress: {idx + 1}/{total_matches} matches processed")
    
    if all_events_dfs:
        print("\nCombining all events...")
        all_events = pd.concat(all_events_dfs, ignore_index=True)
        all_events = ensure_coordinate_columns(all_events)
        all_events = all_events.sort_values(['match_id', 'period', 'minute', 'second', 'index']).reset_index(drop=True)
        print(f"Raw events collected: {len(all_events)}")

        combined_path = COMBINED_DATA_PATH / "all_events_combined.parquet"
        all_events.to_parquet(combined_path, index=False)
        print(f"Combined raw-event file saved to: {combined_path}")
        
        return all_events
    else:
        print("\nNo event data collected")
        return pd.DataFrame()


if not all_matches.empty:
    match_ids = all_matches['match_id'].tolist()
    print(f"\nMatches: {len(match_ids)}")
    
    all_events = collect_event_data(match_ids)
else:
    print("No matches available for event collection")
    all_events = pd.DataFrame()


Matches: 1517
Found 1551 existing event files.



Progress: 50/1517 matches processed



Progress: 100/1517 matches processed



Progress: 150/1517 matches processed



Progress: 200/1517 matches processed



Progress: 250/1517 matches processed


... (23 outputs left out)


Progress: 1450/1517 matches processed



Progress: 1500/1517 matches processed

Combining all events...


Raw events collected: 5321459


Combined raw-event file saved to: /Users/gabrielcarbinatto/Documents/GitHub/Synergy-VAEP/data/raw/combined/all_events_combined.parquet


## **5. Export and audit**

| file | content |
|---|---|
| `data/raw/matches/matches_competition_{c}_season_{s}.parquet` | matches of one competition and season |
| `data/raw/events/events_match_{id}.parquet` | events of one match |
| `data/raw/combined/all_matches_combined.parquet` | all matches |
| `data/raw/combined/all_events_combined.parquet` | all events, with split coordinates |

The audit checks one match file per competition–season, one event file per match, and the uniqueness of event identifiers.

In [6]:
match_files = sorted(MATCHES_PATH.glob("matches_competition_*.parquet"))
event_files = sorted(EVENTS_PATH.glob("events_match_*.parquet"))
event_match_ids = {int(path.stem.rsplit("_", 1)[-1]) for path in event_files}

audit = pd.DataFrame([
    {"metric": "competition_seasons", "value": len(COMPETITIONS_SEASONS)},
    {"metric": "match_files", "value": len(match_files)},
    {"metric": "matches", "value": len(all_matches)},
    {"metric": "event_files", "value": len(event_files)},
    {"metric": "events", "value": len(all_events)},
])
display(audit)

# Checks: failures indicate missing files; run the collection again to complete them.
assert len(match_files) >= len(COMPETITIONS_SEASONS), "Missing match file"
assert set(all_matches["match_id"]) <= event_match_ids, "Missing events for at least one match"
assert all_events["id"].is_unique, "Duplicate event identifier"

,metric,value
0,competition_seasons,4
1,match_files,5
2,matches,1517
3,event_files,1551
4,events,5321459
